In [ ]:
import numpy as np
import pandas as pd

from Bio import SeqIO
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.linalg import eigs


# =====================================================
# Load AAindex 158
# =====================================================

AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"


def load_AAindex(file):

    df = pd.read_csv(file)

    aa_features = {}

    for aa in AA_ORDER:

        aa_features[aa] = (
            df[aa]
            .values
            .astype(float)
        )

    return aa_features



# =====================================================
# coordinate.m
# =====================================================

def coordinate():

    P=[]

    for i in range(1,21):

        P.append([
            np.cos(i*2*np.pi/20),
            np.sin(i*2*np.pi/20),
            1
        ])

    P=np.array(P)


    V={}

    for i in range(20):

        for j in range(20):

            V[(i,j)] = (
                P[i]
                +
                0.25*(P[j]-P[i])
            )


    return P,V



# =====================================================
# GRS.m
# =====================================================

def GRS(seq, P, V):


    results=[]


    M = AA_ORDER


    for prop in range(158):


        c=[np.array([0,0,0])]

        d=np.zeros(3)

        y=np.zeros(20)


        for i,aa in enumerate(seq):


            x=np.zeros(20)

            if aa in M:

                x[M.index(aa)] = 1



            if i==0:


                c.append(
                    c[-1]
                    +
                    x@P
                )


            else:


                if np.sum(x)==0:

                    d=d*(i-1)/i

                    c.append(
                        c[-1]
                        +
                        np.array([0,0,1])
                        +
                        d
                    )


                elif np.sum(y)==0:


                    d=d*(i-1)/i

                    c.append(
                        c[-1]
                        +
                        x@P
                        +
                        d
                    )


                else:


                    old=np.where(y==1)[0][0]
                    new=np.where(x==1)[0][0]


                    d = (
                        d*(i-1)/i
                        +
                        V[(old,new)]/i
                    )


                    c.append(
                        c[-1]
                        +
                        x@P
                        +
                        d
                    )


            y=x



        results.append(
            np.array(c)
        )


    return results



# =====================================================
# ME.m
# =====================================================

def ME(W):


    # remove first row

    W=W[1:]


    if len(W)<2:
        return 0



    E=squareform(
        pdist(W)
    )


    x=len(W)


    sdist=np.zeros(
        (x,x)
    )


    for i in range(x):

        for j in range(i,x):

            if j-i==1:

                sdist[i,j]=E[i,j]


            elif j-i>1:

                sdist[i,j]=(
                    sdist[i,j-1]
                    +
                    E[j-1,j]
                )



    sd=sdist+sdist.T


    sdd=sd+np.eye(x)


    L=E/sdd


    value=eigs(
        L,
        k=1,
        which='LM'
    )[0][0]


    return np.real(value)/x



# =====================================================
# SAD.m
# AAC + DPC
# =====================================================

def SAD(seq):


    length=len(seq)


    AAC=[]


    for aa in AA_ORDER:

        AAC.append(
            seq.count(aa)/length
        )



    DPC=np.zeros(
        (20,20)
    )


    if length>1:


        for i,a1 in enumerate(AA_ORDER):

            for j,a2 in enumerate(AA_ORDER):


                count=0


                for k in range(length-1):

                    if (
                        seq[k]==a1
                        and
                        seq[k+1]==a2
                    ):

                        count+=1


                DPC[i,j]=(
                    count/(length-1)
                )


    return (
        np.array(AAC),
        DPC.flatten()
    )



# =====================================================
# Complete FEGS
# =====================================================

def FEGS(sequence, P,V):


    seq=sequence.upper()



    # EL 158

    graph=GRS(
        seq,
        P,
        V
    )


    EL=[]


    for g in graph:

        EL.append(
            ME(g)
        )


    EL=np.array(EL)



    # AAC + DPC

    FA,FD=SAD(seq)



    FV=np.concatenate(
        [
            EL,
            FA,
            FD
        ]
    )


    return FV



# =====================================================
# FASTA batch extraction
# =====================================================

def fasta_to_FEGS(
        fasta_file,
        output_file,
        aaindex_file
):


    aa_features=load_AAindex(
        aaindex_file
    )


    P,V=coordinate()


    results=[]


    for record in SeqIO.parse(
        fasta_file,
        "fasta"
    ):

        seq=str(record.seq)


        feat=FEGS(
            seq,
            P,
            V
        )


        row={
            "ID":record.id,
            "Sequence":seq
        }


        for i,x in enumerate(feat):

            row[
                f"FEGS_{i+1}"
            ]=x



        results.append(row)



    df=pd.DataFrame(results)


    df.to_csv(
        output_file,
        index=False
    )


    print("Samples:",len(df))
    print("Features:",len(df.columns)-2)



# =====================================================
# RUN
# =====================================================


fasta_to_FEGS(
    fasta_file="/content/drive/MyDrive/Colab Notebooks/Train_positive_sites.fasta",
    output_file="FEGS_Train_features.csv",
    aaindex_file="/content/AAindex_158.csv"
)